# Classwork 8
## Setting Up Inputs and generating a list of counties

In [32]:
import arcpy # ArcGIS Python library
import os # for operating system interactions
import shutil # for file operations
import pathlib # for filesystem paths

In [33]:
# Extract the folder in the same directory as this script
# input file geodatabase path
input_fgdb = r".\Chapter 10\Chapter 10.gdb"
# input feature class name
input_fc_name = "Highways_Intersect"
# output folder
output_folder = r".\zipped_outputs"

In [34]:
# combine to get full path to input feature class
full_fc_path = os.path.join(
    input_fgdb, input_fc_name
)

In [35]:
 # check if the input feature class exists

arcpy.Exists(full_fc_path)

True

In [36]:
counties = arcpy.da.SearchCursor(full_fc_path, "NAMELSAD") # get list of county names

# display number of counties

counties_list = [row[0] for row in counties]
print(f"Number of counties: {len(counties_list)}")

Number of counties: 6191


In [37]:
counties = list(set(counties_list))  # get unique county names
counties.sort() # sort the counties alphabetically
counties # display the sorted list of counties

['Alameda County',
 'Alpine County',
 'Amador County',
 'Butte County',
 'Colusa County',
 'Contra Costa County',
 'Del Norte County',
 'El Dorado County',
 'Fresno County',
 'Glenn County',
 'Humboldt County',
 'Imperial County',
 'Inyo County',
 'Kern County',
 'Kings County',
 'Lake County',
 'Lassen County',
 'Los Angeles County',
 'Madera County',
 'Marin County',
 'Mariposa County',
 'Mendocino County',
 'Merced County',
 'Modoc County',
 'Mono County',
 'Monterey County',
 'Napa County',
 'Nevada County',
 'Orange County',
 'Placer County',
 'Riverside County',
 'Sacramento County',
 'San Benito County',
 'San Bernardino County',
 'San Diego County',
 'San Francisco County',
 'San Joaquin County',
 'San Luis Obispo County',
 'San Mateo County',
 'Santa Barbara County',
 'Santa Clara County',
 'Santa Cruz County',
 'Shasta County',
 'Sierra County',
 'Siskiyou County',
 'Solano County',
 'Sonoma County',
 'Stanislaus County',
 'Sutter County',
 'Tehama County',
 'Trinity County',

In [38]:
# create output folder if it doesn't exist

if not os.path.exists(output_folder):
    os.mkdir(output_folder)

In [39]:
county = counties[0] # access the first county
county # display the county name

'Alameda County'

In [40]:
# remove spaces from county name and replace with underscores
county_no_spaces = county.replace(" ", "_")
county_no_spaces

'Alameda_County'

In [41]:
# create a file geodatabase for the county
fgdb = arcpy.management.CreateFileGDB(
    out_folder_path = output_folder,
    out_name=f"{county_no_spaces}_Output"
)

fgdb

ExecuteError: ERROR 000258: Output c:\Users\arjav\DevSource\sgsup-arjav-222\homework\week13\zipped_outputs\Alameda_County_Output.gdb already exists
Failed to execute (CreateFileGDB).


In [42]:
    # display the path to the created file geodatabase

fgdb[0]

'.\\zipped_outputs\\Alameda_County_Output.gdb'

In [43]:
# export features for the county into the file geodatabase
output_fc = arcpy.conversion.ExportFeatures(
    in_features=full_fc_path,
    out_features=os.path.join(
        fgdb[0],
        f"{county_no_spaces}_Highways"),
        where_clause=f"NAMELSAD = '{county}'"
)
output_fc[0]

ExecuteError: Failed to execute. Parameters are not valid.
ERROR 000725: Output Feature Class: Dataset .\zipped_outputs\Alameda_County_Output.gdb\Alameda_County_Highways already exists.
Failed to execute (ExportFeatures).


In [44]:
    # get and display the number of features exported

arcpy.management.GetCount(output_fc[0])

<Result '427'>

In [56]:
# get the path for the file geodatabase you created
source_fgdb_path = pathlib.Path(fgdb[0])
# The name of the folder to place the File Geodatabase in
fgdb_folder_name = source_fgdb_path.stem
# The location of the folder to place the File Geodatabase in
fgdb_folder_location = source_fgdb_path.parent
# The path to the folder to place the File Geodatabase in
fgdb_folder_path = fgdb_folder_location.joinpath(fgdb_folder_name)
# The path of our copied File Geodatabase
fgdb_path = fgdb_folder_path.joinpath(source_fgdb_path.name)

In [58]:
# Print the path to the copied File Geodatabase
fgdb_path

WindowsPath('zipped_outputs/Alameda_County_Output/Alameda_County_Output.gdb')

In [59]:
# Copy the File Geodatabase, ignoring lock files
shutil.copytree(
    source_fgdb_path,
    fgdb_path,
    ignore=shutil.ignore_patterns('*.lock')
    )

WindowsPath('zipped_outputs/Alameda_County_Output/Alameda_County_Output.gdb')

In [60]:
# Zip the File Geodatabase
zipped_fgdb = shutil.make_archive(
    base_name=fgdb_folder_path, # The name of the archive, not including the file extension
    format='zip', # The archive format
    root_dir=fgdb_folder_path, # The directory to archive
)


In [65]:
# delete the original file geodatabase
arcpy.env.overwriteOutput = True
arcpy.management.Delete(fgdb)

<Result 'true'>

In [66]:
# delete the folder and its contents
shutil.rmtree(fgdb_folder_path)

## Create a Repeatable Function

In [67]:
def zip_county_highways(full_fc_path, output_folder, county):
    # remove spaces from county name
    county_no_spaces = county.replace(" ", "_")
    # create a file geodatabase
    fgdb = arcpy.management.CreateFileGDB(
        out_folder_path=output_folder,
        out_name=f"{county_no_spaces}_Output"
    )
    # Create a feature class
    output_fc = arcpy.conversion.ExportFeatures(
        in_features=full_fc_path,
        out_features=os.path.join(
            fgdb[0],
            f"{county_no_spaces}_Highways"),
            where_clause=f"NAMELSAD = '{county}'"
    )
    source_fgdb_path = pathlib.Path(fgdb[0])
    # The name of the folder to place the File Geodatabase in
    fgdb_folder_name = source_fgdb_path.stem
    # The location of the folder to place the File Geodatabase in
    fgdb_folder_location = source_fgdb_path.parent
    # The path to the folder to place the File Geodatabase in
    fgdb_folder_path = fgdb_folder_location.joinpath(fgdb_folder_name)
    # The path of our copied File Geodatabase
    fgdb_path = fgdb_folder_path.joinpath(source_fgdb_path.name)
    # copy the file geodatabase into a temp folder
    shutil.copytree(
        source_fgdb_path,
        fgdb_path,
        ignore=shutil.ignore_patterns('*.lock')
        )
    # Zip the File Geodatabase
    zipped_fgdb = shutil.make_archive(
        base_name=fgdb_folder_path, # The name of the archive, not including the file extension
        format='zip', # The archive format
        root_dir=fgdb_folder_path, # The directory to archive
    )
    # delete the file geodatabase
    arcpy.management.Delete(fgdb)
    # remove the temp folder
    shutil.rmtree(fgdb_folder_path)
    # return the zip file path
    return zipped_fgdb

In [68]:
zip_county_highways(full_fc_path, output_folder, "Butte County")

'c:\\Users\\arjav\\DevSource\\sgsup-arjav-222\\homework\\week13\\zipped_outputs\\Butte_County_Output.zip'